# MLP Fundamentals: Backprop, Initialization, BatchNorm & Dropout

*Source:* `~/Projects/school/tamu-grad/ecen740/Copy_of_ECEN740_Project1.ipynb` (ECEN 740, Project 1).

A guided tour of multilayer-perceptron mechanics on image data: per-channel
normalization, fixing a broken linear classifier, adding hidden layers, a from-scratch
multi-class **hinge loss** vs cross-entropy, **manual backprop** through sigmoid and a
two-layer net, the **dead-ReLU** problem, activation comparisons, **Xavier vs Kaiming**
initialization, **BatchNorm**, the train/eval-mode bug, a **dropout** sweep, and a final
best-MLP design.


!!! note "Saved outputs; plots stripped to keep the file light"
    This notebook was **not** re-executed for the textbook (outputs are from the original
    GPU run, `device = 'cuda'`). To keep the committed file well under 2 MB, the embedded
    plot images were removed -- text/printed outputs remain, and every plot regenerates
    when you run the notebook locally.


---
## Part 0: Setup and Imports

Run this cell first. It imports everything you need.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
import numpy as np
import matplotlib.pyplot as plt
import time
import copy
import math
from collections import defaultdict

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

CIFAR10_CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                   'dog', 'frog', 'horse', 'ship', 'truck']

Using device: cuda
GPU: Tesla T4


In [ ]:

def train_one_epoch(model, trainloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0; correct = 0; total = 0
    for inputs, labels in trainloader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    return running_loss / total, 100.0 * correct / total

def evaluate(model, testloader, criterion, device):
    model.eval()
    running_loss = 0.0; correct = 0; total = 0
    with torch.no_grad():
        for inputs, labels in testloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    return running_loss / total, 100.0 * correct / total

def train_and_log(model, trainloader, testloader, criterion, optimizer,
                  epochs, device, scheduler=None, verbose=True):
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'lr': []}
    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch(model, trainloader, criterion, optimizer, device)
        val_loss, val_acc = evaluate(model, testloader, criterion, device)
        current_lr = optimizer.param_groups[0]['lr']
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['lr'].append(current_lr)
        if scheduler is not None:
            scheduler.step()
        if verbose and (epoch % max(1, epochs // 10) == 0 or epoch == epochs - 1):
            print(f'Epoch {epoch+1:3d}/{epochs} | '
                  f'Train Loss {train_loss:.4f} Acc {train_acc:.1f}% | '
                  f'Val Loss {val_loss:.4f} Acc {val_acc:.1f}% | '
                  f'LR {current_lr:.6f}')
    return history

def plot_history(history, title='Training History'):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(history['train_loss'], label='Train Loss')
    ax1.plot(history['val_loss'], label='Val Loss')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
    ax1.set_title(f'{title} — Loss'); ax1.legend(); ax1.grid(True, alpha=0.3)
    ax2.plot(history['train_acc'], label='Train Acc')
    ax2.plot(history['val_acc'], label='Val Acc')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
    ax2.set_title(f'{title} — Accuracy'); ax2.legend(); ax2.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

def plot_multiple_histories(histories, labels, title='Comparison'):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    for h, label in zip(histories, labels):
        ax1.plot(h['train_loss'], label=f'{label}')
        ax2.plot(h['val_acc'], label=f'{label}')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Train Loss')
    ax1.set_title(f'{title} — Training Loss'); ax1.legend(); ax1.grid(True, alpha=0.3)
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Val Accuracy (%)')
    ax2.set_title(f'{title} — Validation Accuracy'); ax2.legend(); ax2.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

print('Utilities loaded.')

Utilities loaded.


---
## Part 1: Data Exploration and Preprocessing

Before building any model, you must understand your data. CIFAR-10 consists of 60,000 32×32 color images in 10 classes.

In [ ]:
transform_basic = transforms.Compose([transforms.ToTensor()])
trainset_raw = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_basic)
testset_raw = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_basic)
print(f'Training set size: {len(trainset_raw)}')
print(f'Test set size: {len(testset_raw)}')
print(f'Image shape: {trainset_raw[0][0].shape}')

100%|██████████| 170M/170M [00:43<00:00, 3.96MB/s]


Training set size: 50000
Test set size: 10000
Image shape: torch.Size([3, 32, 32])


### Task 1.1: Visualize the data

Display a grid of 40 random images (4 per class) from the training set. Each image should have its class label as the title. Use `plt.subplots` to create a 4×10 grid.

**Important:** Call `plt.show()` at the end so the figure is finalized before the test cell runs.

In [ ]:
# Create a 4x10 grid showing 4 random samples from each of the 10 classes.
# Hint: use plt.subplots(4, 10, figsize=(20, 8))
# IMPORTANT: End your code with plt.show()

fig, axes = plt.subplots(4, 10, figsize=(20, 8))
labels = np.array([label for _, label in trainset_raw])

for i in range(10):  # 10 classes = columns
    class_indices = np.where(labels == i)[0]
    samples = np.random.choice(class_indices, 4, replace=False)

    for j, idx in enumerate(samples):  # 4 samples = rows
        img, _ = trainset_raw[idx]
        axes[j, i].imshow(img.permute(1, 2, 0))
        axes[j, i].axis('off')
    axes[0, i].set_title(CIFAR10_CLASSES[i], fontsize=10)

plt.show()
plt.figure(fig.number)


<Figure size 2000x800 with 40 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

In [ ]:
fig = plt.gcf()
n_axes = len(fig.axes)
if n_axes == 0:
    all_figs = [plt.figure(i) for i in plt.get_fignums()]
    if all_figs:
        fig = all_figs[-1]
        n_axes = len(fig.axes)
assert n_axes >= 10, (
    f'Expected at least 10 subplots in your grid, got {n_axes}. '
    f'Make sure you use plt.subplots(4, 10, ...) and do NOT call plt.close().'
)
print(f'Test 1.1 PASSED: Grid visualization created with {n_axes} subplots.')

AssertionError: Expected at least 10 subplots in your grid, got 0. Make sure you use plt.subplots(4, 10, ...) and do NOT call plt.close().

<Figure size 640x480 with 0 Axes>

### Test 1.1 explanation

The provided Test 1.1 failed with the message that the current figure had `0` axes, even though my visualization cell creates a `4 x 10` subplot grid with `plt.subplots(4, 10, figsize=(20, 8))` and ends with `plt.show()` as instructed.

I believe my implementation is correct for the following reasons:

1. It displays **40 total training images**, with **4 random samples from each of the 10 CIFAR-10 classes**.
2. Each subplot is labeled with the corresponding class name.
3. The code uses `plt.subplots(4, 10, ...)` as required.
4. The code does **not** call `plt.close()` or intentionally remove axes.
5. The figure renders correctly when I run the visualization cell itself.

Because the plot appears correctly but the test still reports `0` axes, I believe this is due to a notebook figure-state issue rather than a problem in the plotting logic. For that reason, I believe my solution is correct and satisfies the requirements of Task 1.1.

### Task 1.2: Compute per-channel statistics

Compute the mean and std of the entire training set for each of the 3 color channels (R, G, B). Store them in `channel_means` and `channel_stds` as tuples of length 3.

In [ ]:

all_imgs = torch.stack([img for img, _ in trainset_raw])
channel_means = tuple(all_imgs.mean(dim=(0,2,3)).tolist())
channel_stds = tuple(all_imgs.std(dim=(0,2,3)).tolist())


print(f'Channel means: {channel_means}')
print(f'Channel stds: {channel_stds}')

Channel means: (0.491400808095932, 0.48215898871421814, 0.44653093814849854)
Channel stds: (0.24703224003314972, 0.24348513782024384, 0.26158785820007324)


In [ ]:
assert channel_means is not None and channel_stds is not None
assert len(channel_means) == 3 and len(channel_stds) == 3
assert abs(channel_means[0] - 0.4914) < 0.01, f'Red channel mean looks off: {channel_means[0]:.4f}'
assert abs(channel_stds[0] - 0.2470) < 0.02, f'Red channel std looks off: {channel_stds[0]:.4f}'
print('Test 1.2 PASSED: Channel statistics are correct.')

Test 1.2 PASSED: Channel statistics are correct.


### Task 1.3: Create data loaders with normalization and augmentation

Create two transform pipelines:
1. `transform_train`: RandomCrop(32, padding=4), RandomHorizontalFlip, ToTensor, Normalize
2. `transform_test`: ToTensor, Normalize

Then create datasets and dataloaders with `batch_size=128`.

In [ ]:

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(channel_means, channel_stds)
])


transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(channel_means, channel_stds)
])


trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
trainloader = DataLoader(trainset, batch_size=128, shuffle=True)
testloader = DataLoader(testset, batch_size=128, shuffle=False)



In [ ]:
batch = next(iter(trainloader))
assert batch[0].shape == torch.Size([128, 3, 32, 32]), f'Unexpected batch shape: {batch[0].shape}'
assert batch[0].mean().abs() < 1.0, 'Data does not appear to be normalized.'
print('Test 1.3 PASSED: Data loaders created with normalization.')

Test 1.3 PASSED: Data loaders created with normalization.


---
## Part 2: Your First (Broken) Classifier

In [ ]:
class LinearClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(3 * 32 * 32, 10)
    def forward(self, x):
        return self.fc(self.flatten(x))

model_broken = LinearClassifier().to(device)
optimizer_broken = optim.SGD(model_broken.parameters(), lr=10.0)
criterion = nn.CrossEntropyLoss()
print('Training a linear classifier with lr=10.0 ...')
history_broken = train_and_log(model_broken, trainloader, testloader, criterion,
                               optimizer_broken, epochs=5, device=device)

Training a linear classifier with lr=10.0 ...
Epoch   1/5 | Train Loss 1117.6164 Acc 20.0% | Val Loss 907.7290 Acc 22.7% | LR 10.000000
Epoch   2/5 | Train Loss 1022.4862 Acc 21.3% | Val Loss 1756.7679 Acc 23.0% | LR 10.000000
Epoch   3/5 | Train Loss 1025.7026 Acc 21.4% | Val Loss 874.0939 Acc 23.3% | LR 10.000000
Epoch   4/5 | Train Loss 1037.1110 Acc 21.6% | Val Loss 1053.8269 Acc 26.3% | LR 10.000000
Epoch   5/5 | Train Loss 1029.4161 Acc 21.6% | Val Loss 1194.0973 Acc 24.9% | LR 10.000000


### Task 2.1: Diagnose and fix the linear classifier

The model above likely produced NaN losses or random-chance accuracy. Fix it by choosing a reasonable learning rate, and train for 10 epochs. Store the result in `history_fixed_linear`.

In [ ]:

model_linear = LinearClassifier().to(device)
optimizer_linear = optim.SGD(model_linear.parameters(), lr=0.01)
history_fixed_linear = train_and_log(model_linear, trainloader, testloader, criterion, optimizer_linear, epochs=10, device=device)



Epoch   1/10 | Train Loss 1.9984 Acc 28.5% | Val Loss 1.9183 Acc 34.2% | LR 0.010000
Epoch   2/10 | Train Loss 1.9499 Acc 30.9% | Val Loss 1.8716 Acc 34.7% | LR 0.010000
Epoch   3/10 | Train Loss 1.9437 Acc 31.1% | Val Loss 1.9120 Acc 33.3% | LR 0.010000
Epoch   4/10 | Train Loss 1.9442 Acc 31.2% | Val Loss 1.8370 Acc 36.1% | LR 0.010000
Epoch   5/10 | Train Loss 1.9313 Acc 31.9% | Val Loss 1.8755 Acc 34.5% | LR 0.010000
Epoch   6/10 | Train Loss 1.9338 Acc 31.9% | Val Loss 1.8746 Acc 35.5% | LR 0.010000
Epoch   7/10 | Train Loss 1.9340 Acc 32.1% | Val Loss 1.8725 Acc 35.3% | LR 0.010000
Epoch   8/10 | Train Loss 1.9295 Acc 32.0% | Val Loss 1.8587 Acc 35.5% | LR 0.010000
Epoch   9/10 | Train Loss 1.9328 Acc 31.9% | Val Loss 1.8884 Acc 34.9% | LR 0.010000
Epoch  10/10 | Train Loss 1.9280 Acc 32.2% | Val Loss 1.8836 Acc 35.5% | LR 0.010000


In [ ]:
assert history_fixed_linear is not None
final_acc = history_fixed_linear['val_acc'][-1]
assert final_acc > 20.0, f'Linear classifier should get >20% accuracy, got {final_acc:.1f}%'
assert not any(np.isnan(history_fixed_linear['train_loss'])), 'Training loss contains NaN!'
print(f'Test 2.1 PASSED: Linear classifier achieved {final_acc:.1f}% val accuracy.')

Test 2.1 PASSED: Linear classifier achieved 35.5% val accuracy.


### Task 2.2: Add hidden layers

Build a simple MLP:
- `Flatten → Linear(3072, 512) → ReLU → Linear(512, 256) → ReLU → Linear(256, 10)`

Train for 20 epochs with an appropriate optimizer.

In [ ]:

class SimpleMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(3072, 512), nn.ReLU(),
            nn.Linear(512, 256), nn.ReLU(),
            nn.Linear(256, 10),
        )

    def forward(self, x):
        return self.net(x)

model_mlp = SimpleMLP().to(device)
optimizer_mlp = optim.Adam(model_mlp.parameters(), lr=0.001)

history_mlp = train_and_log(model_mlp, trainloader, testloader, criterion, optimizer_mlp, epochs=20, device=device)



Epoch   1/20 | Train Loss 1.7963 Acc 35.2% | Val Loss 1.6640 Acc 40.0% | LR 0.001000
Epoch   3/20 | Train Loss 1.5734 Acc 43.5% | Val Loss 1.6169 Acc 41.1% | LR 0.001000
Epoch   5/20 | Train Loss 1.4900 Acc 46.6% | Val Loss 1.5783 Acc 43.4% | LR 0.001000
Epoch   7/20 | Train Loss 1.4445 Acc 48.0% | Val Loss 1.5376 Acc 44.4% | LR 0.001000
Epoch   9/20 | Train Loss 1.4112 Acc 49.3% | Val Loss 1.5137 Acc 45.7% | LR 0.001000
Epoch  11/20 | Train Loss 1.3890 Acc 50.0% | Val Loss 1.5034 Acc 46.0% | LR 0.001000
Epoch  13/20 | Train Loss 1.3627 Acc 51.3% | Val Loss 1.5712 Acc 44.6% | LR 0.001000
Epoch  15/20 | Train Loss 1.3451 Acc 51.6% | Val Loss 1.5214 Acc 46.9% | LR 0.001000
Epoch  17/20 | Train Loss 1.3303 Acc 52.5% | Val Loss 1.5295 Acc 46.2% | LR 0.001000
Epoch  19/20 | Train Loss 1.3162 Acc 52.8% | Val Loss 1.5334 Acc 44.2% | LR 0.001000
Epoch  20/20 | Train Loss 1.3134 Acc 52.8% | Val Loss 1.5299 Acc 45.3% | LR 0.001000


In [ ]:
assert history_mlp is not None
mlp_acc = history_mlp['val_acc'][-1]
linear_acc = history_fixed_linear['val_acc'][-1]
assert mlp_acc > linear_acc, f'MLP ({mlp_acc:.1f}%) should outperform linear ({linear_acc:.1f}%)'
assert mlp_acc > 40.0, f'MLP should get >40%, got {mlp_acc:.1f}%'
print(f'Test 2.2 PASSED: MLP ({mlp_acc:.1f}%) outperforms linear ({linear_acc:.1f}%).')
plot_multiple_histories([history_fixed_linear, history_mlp], ['Linear', 'MLP'], 'Linear vs MLP')

Test 2.2 PASSED: MLP (45.3%) outperforms linear (35.5%).


<Figure size 1400x500 with 2 Axes>

### Task 2.3: Count parameters

Write a function `count_parameters(model)` that returns the total number of trainable parameters.

In [ ]:

def count_parameters(model):
    return sum(p.numel() for p in model.parameters())


print(f'LinearClassifier: {count_parameters(LinearClassifier()):,} parameters')
print(f'SimpleMLP: {count_parameters(SimpleMLP()):,} parameters')

LinearClassifier: 30,730 parameters
SimpleMLP: 1,707,274 parameters


In [ ]:
n_params = count_parameters(SimpleMLP())
assert 1_500_000 < n_params < 2_000_000, f'SimpleMLP should have ~1.7M params, got {n_params:,}'
print(f'Test 2.3 PASSED: SimpleMLP has {n_params:,} parameters.')

Test 2.3 PASSED: SimpleMLP has 1,707,274 parameters.


---
## Part 3: Loss Functions: Cross-Entropy vs. Hinge

### Task 3.1: Implement multi-class hinge loss

Implement `MultiClassHingeLoss` as a PyTorch module. For a single sample with scores **s** and correct class *y*:

$$L = \sum_{j \neq y} \max(0, s_j - s_y + \Delta)$$

where Δ is the margin (default 1.0). The batch loss should be the mean over samples.

In [ ]:

class MultiClassHingeLoss(nn.Module):
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin

    def forward(self, scores, labels):
        correct_scores = scores[torch.arange(scores.size(0)), labels].unsqueeze(1)
        margins = torch.clamp(scores - correct_scores + self.margin, min=0)
        margins[torch.arange(scores.size(0)), labels] = 0
        return margins.sum(dim=1).mean()



In [ ]:
hinge_loss_fn = MultiClassHingeLoss(margin=1.0)
test_scores = torch.tensor([[3.2, 5.1, -1.7]])
test_labels = torch.tensor([0])
loss_val = hinge_loss_fn(test_scores, test_labels).item()
assert abs(loss_val - 2.9) < 0.01, f'Expected 2.9, got {loss_val:.4f}'
test_scores2 = torch.tensor([[10.0, 1.0, 1.0]])
loss_val2 = hinge_loss_fn(test_scores2, torch.tensor([0])).item()
assert abs(loss_val2) < 0.01, f'Expected 0, got {loss_val2:.4f}'
print('Test 3.1 PASSED: Hinge loss implementation correct.')

Test 3.1 PASSED: Hinge loss implementation correct.


### Task 3.2: Train with both losses and compare

Train two identical `SimpleMLP` models for 20 epochs:
- One with `nn.CrossEntropyLoss`
- One with your `MultiClassHingeLoss`

Use the same optimizer (`SGD`, lr=0.01, momentum=0.9) for both.

In [ ]:
model_ce = SimpleMLP().to(device)
optimizer_ce = optim.SGD(model_ce.parameters(), lr=0.01, momentum=0.9)
history_ce = train_and_log(model_ce, trainloader, testloader, nn.CrossEntropyLoss(), optimizer_ce, epochs=20, device=device)

model_hinge = SimpleMLP().to(device)
optimizer_hinge = optim.SGD(model_hinge.parameters(), lr=0.01, momentum=0.9)
history_hinge = train_and_log(model_hinge, trainloader, testloader, MultiClassHingeLoss(), optimizer_hinge, epochs=20, device=device)




Epoch   1/20 | Train Loss 1.8418 Acc 33.4% | Val Loss 1.6534 Acc 41.2% | LR 0.010000
Epoch   3/20 | Train Loss 1.5732 Acc 43.6% | Val Loss 1.5015 Acc 46.5% | LR 0.010000
Epoch   5/20 | Train Loss 1.4862 Acc 46.4% | Val Loss 1.4704 Acc 48.0% | LR 0.010000
Epoch   7/20 | Train Loss 1.4353 Acc 48.4% | Val Loss 1.4371 Acc 49.4% | LR 0.010000
Epoch   9/20 | Train Loss 1.3942 Acc 49.8% | Val Loss 1.4257 Acc 49.2% | LR 0.010000
Epoch  11/20 | Train Loss 1.3636 Acc 51.1% | Val Loss 1.4356 Acc 48.7% | LR 0.010000
Epoch  13/20 | Train Loss 1.3325 Acc 52.2% | Val Loss 1.4061 Acc 50.3% | LR 0.010000
Epoch  15/20 | Train Loss 1.3145 Acc 52.9% | Val Loss 1.3985 Acc 50.4% | LR 0.010000
Epoch  17/20 | Train Loss 1.2919 Acc 53.4% | Val Loss 1.3661 Acc 51.4% | LR 0.010000
Epoch  19/20 | Train Loss 1.2719 Acc 54.3% | Val Loss 1.3747 Acc 51.0% | LR 0.010000
Epoch  20/20 | Train Loss 1.2684 Acc 54.3% | Val Loss 1.3611 Acc 51.9% | LR 0.010000
Epoch   1/20 | Train Loss 5.2900 Acc 30.8% | Val Loss 4.8710 Acc 

In [ ]:
assert history_ce is not None and history_hinge is not None
assert len(history_ce['val_acc']) == 20
print(f'Cross-Entropy final val acc: {history_ce["val_acc"][-1]:.1f}%')
print(f'Hinge Loss final val acc: {history_hinge["val_acc"][-1]:.1f}%')
plot_multiple_histories([history_ce, history_hinge],
                       ['Cross-Entropy', 'Hinge Loss'], 'Loss Function Comparison')
print('Test 3.2 PASSED.')

Cross-Entropy final val acc: 51.9%
Hinge Loss final val acc: 31.1%


<Figure size 1400x500 with 2 Axes>

Test 3.2 PASSED.


### Task 3.3: Visualize the confidence distributions

After training, run both models on the test set. For each model, collect the softmax probability assigned to the **true class** for all test samples. Plot two histograms side by side.

In [ ]:
# For each model (model_ce, model_hinge):
# 1. Run inference on testloader
# 2. Apply softmax to outputs
# 3. Collect P(true_class) for each sample
# 4. Plot two histograms side by side

def get_true_class_probs(model, dataloader, device):
    model.eval()
    probs = []
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = torch.softmax(model(inputs), dim=1)
            true_probs = outputs[torch.arange(len(labels)), labels]
            probs.append(true_probs.cpu())
    return torch.cat(probs).numpy()

probs_ce = get_true_class_probs(model_ce, testloader, device)
probs_hinge = get_true_class_probs(model_hinge, testloader, device)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.hist(probs_ce, bins=50, range=(0, 1))
ax1.set_title('Cross-Entropy'); ax1.set_xlabel('P(true class)')
ax2.hist(probs_hinge, bins=50, range=(0, 1))
ax2.set_title('Hinge Loss'); ax2.set_xlabel('P(true class)')
plt.tight_layout()
plt.show()


<Figure size 1400x500 with 2 Axes>

---
## Part 4: Optimizers: SGD, Momentum, and Adam

### Task 4.1: Optimizer comparison experiment

Train three identical `SimpleMLP` models for 20 epochs each:
1. **SGD** (lr=0.01, no momentum)
2. **SGD with Momentum** (lr=0.01, momentum=0.9)
3. **Adam** (lr=0.001)

Use cross-entropy loss for all three.

In [ ]:

model_sgd = SimpleMLP().to(device)
history_sgd = train_and_log(model_sgd, trainloader, testloader, nn.CrossEntropyLoss(),
                            optim.SGD(model_sgd.parameters(), lr=0.01), epochs=20, device=device)

model_mom = SimpleMLP().to(device)
history_momentum = train_and_log(model_mom, trainloader, testloader, nn.CrossEntropyLoss(),
                                 optim.SGD(model_mom.parameters(), lr=0.01, momentum=0.9), epochs=20, device=device)

model_adam = SimpleMLP().to(device)
history_adam = train_and_log(model_adam, trainloader, testloader, nn.CrossEntropyLoss(),
                             optim.Adam(model_adam.parameters(), lr=0.001), epochs=20, device=device)



Epoch   1/20 | Train Loss 2.0705 Acc 25.6% | Val Loss 1.9485 Acc 32.8% | LR 0.010000
Epoch   3/20 | Train Loss 1.7977 Acc 36.1% | Val Loss 1.7569 Acc 38.7% | LR 0.010000
Epoch   5/20 | Train Loss 1.6913 Acc 39.5% | Val Loss 1.6657 Acc 41.1% | LR 0.010000
Epoch   7/20 | Train Loss 1.6211 Acc 42.3% | Val Loss 1.6207 Acc 42.4% | LR 0.010000
Epoch   9/20 | Train Loss 1.5729 Acc 44.2% | Val Loss 1.5735 Acc 44.1% | LR 0.010000
Epoch  11/20 | Train Loss 1.5351 Acc 45.4% | Val Loss 1.5330 Acc 46.0% | LR 0.010000
Epoch  13/20 | Train Loss 1.5038 Acc 46.6% | Val Loss 1.5157 Acc 46.2% | LR 0.010000
Epoch  15/20 | Train Loss 1.4752 Acc 47.4% | Val Loss 1.4813 Acc 47.6% | LR 0.010000
Epoch  17/20 | Train Loss 1.4524 Acc 48.1% | Val Loss 1.4679 Acc 48.1% | LR 0.010000
Epoch  19/20 | Train Loss 1.4329 Acc 48.9% | Val Loss 1.4780 Acc 47.6% | LR 0.010000
Epoch  20/20 | Train Loss 1.4257 Acc 49.2% | Val Loss 1.4456 Acc 49.2% | LR 0.010000
Epoch   1/20 | Train Loss 1.8429 Acc 33.4% | Val Loss 1.6640 Acc 

In [ ]:
assert all(h is not None for h in [history_sgd, history_momentum, history_adam])
sgd_acc = history_sgd['val_acc'][-1]
mom_acc = history_momentum['val_acc'][-1]
adam_acc = history_adam['val_acc'][-1]
print(f'SGD: {sgd_acc:.1f}%  Momentum: {mom_acc:.1f}%  Adam: {adam_acc:.1f}%')
assert mom_acc > sgd_acc - 5, 'Momentum should generally improve over plain SGD.'
plot_multiple_histories([history_sgd, history_momentum, history_adam],
                       ['SGD', 'SGD+Momentum', 'Adam'], 'Optimizer Comparison')
print('Test 4.1 PASSED.')

SGD: 49.2%  Momentum: 52.2%  Adam: 46.3%


<Figure size 1400x500 with 2 Axes>

Test 4.1 PASSED.


### Task 4.2: Visualize optimizer trajectories on a 2D loss surface

Optimize a simple 2D function:

$$f(x, y) = 0.1 \cdot x^2 + 2.0 \cdot y^2$$

This is a long, narrow valley (eigenvalues differ by 20×). Starting from (5, 5), run 50 steps of each optimizer and plot trajectories on a contour plot.

**Steps:**
1. Implement `run_optimizer_2d` that minimizes the function and returns `(x, y)` positions
2. Run with SGD (lr=0.1), SGD+Momentum (lr=0.1, momentum=0.9), and Adam (lr=0.5)
3. Plot trajectories on top of a contour plot

In [ ]:

def run_optimizer_2d(optimizer_class, optimizer_kwargs, start=(5.0, 5.0), steps=50):
    """Minimize f(x,y) = 0.1*x^2 + 2*y^2 starting from `start`.
    Returns: list of (x, y) positions at each step."""
    x = torch.tensor([start[0]], requires_grad=True)
    y = torch.tensor([start[1]], requires_grad=True)
    optimizer = optimizer_class([x, y], **optimizer_kwargs)
    trajectory = [(x.item(), y.item())]
    for _ in range(steps):
      optimizer.zero_grad()
      loss = 0.1 * x**2 + 2.0*y**2
      loss.backward()
      optimizer.step()
      trajectory.append((x.item(), y.item()))
    return trajectory

traj_sgd = run_optimizer_2d(optim.SGD, {'lr': 0.1})
traj_mom = run_optimizer_2d(optim.SGD, {'lr': 0.1, 'momentum': 0.9})
traj_adam = run_optimizer_2d(optim.Adam, {'lr': 0.5})

# Plot contour + trajectories
fig, ax = plt.subplots(figsize=(10, 8))
xs = np.linspace(-6, 6, 200)
ys = np.linspace(-6, 6, 200)
X, Y = np.meshgrid(xs, ys)
Z = 0.1 * X**2 + 2.0 * Y**2
ax.contour(X, Y,Z , levels=30)

for traj, label, color in [(traj_sgd, 'SGD', 'red'), (traj_mom, 'Momentum', 'blue'), (traj_adam, 'Adam', 'green')]:
    pts = np.array(traj)
    ax.plot(pts[:, 0], pts[:, 1], '-o', label=label, color=color, markersize=3)

ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('Optimizer Trajectories on f(x,y) = 0.1x² + 2y²')
ax.legend()
plt.tight_layout()
plt.show()



<Figure size 1000x800 with 1 Axes>

In [ ]:
assert traj_sgd is not None and len(traj_sgd) > 10
sgd_final = np.sqrt(traj_sgd[-1][0]**2 + traj_sgd[-1][1]**2)
mom_final = np.sqrt(traj_mom[-1][0]**2 + traj_mom[-1][1]**2)
adam_final = np.sqrt(traj_adam[-1][0]**2 + traj_adam[-1][1]**2)
print(f'Final distance from origin — SGD: {sgd_final:.4f}, Momentum: {mom_final:.4f}, Adam: {adam_final:.4f}')
print('Test 4.2 PASSED.')

Final distance from origin — SGD: 1.8208, Momentum: 0.3999, Adam: 0.0341
Test 4.2 PASSED.


---
## Part 5: Backpropagation by Hand

### Task 5.1: Verify backpropagation through a sigmoid neuron

Consider: $f = \sigma(w_0 x_0 + w_1 x_1 + w_2)$ with values:
- $w_0 = 2,\ x_0 = -1,\ w_1 = -3,\ x_1 = -2,\ w_2 = -3$

Use PyTorch autograd to compute all five first-order gradients and print them.

In [ ]:
w0 = torch.tensor(2.0, requires_grad=True)
x0 = torch.tensor(-1.0, requires_grad=True)
w1 = torch.tensor(-3.0, requires_grad=True)
x1 = torch.tensor(-2.0, requires_grad=True)
w2 = torch.tensor(-3.0, requires_grad=True)

f = torch.sigmoid(w0*x0 + w1*x1 + w2)
f.backward()

print(f'df/dw0 = {w0.grad.item():.4f}')
print(f'df/dx0 = {x0.grad.item():.4f}')
print(f'df/dw1 = {w1.grad.item():.4f}')
print(f'df/dx1 = {x1.grad.item():.4f}')
print(f'df/dw2 = {w2.grad.item():.4f}')



df/dw0 = -0.1966
df/dx0 = 0.3932
df/dw1 = -0.3932
df/dx1 = -0.5898
df/dw2 = 0.1966


In [ ]:
w0 = torch.tensor(2.0, requires_grad=True)
x0 = torch.tensor(-1.0, requires_grad=True)
w1 = torch.tensor(-3.0, requires_grad=True)
x1 = torch.tensor(-2.0, requires_grad=True)
w2 = torch.tensor(-3.0, requires_grad=True)
f = torch.sigmoid(w0*x0 + w1*x1 + w2)
f.backward()
assert abs(w0.grad.item() - (-0.197)) < 0.01, f'df/dw0 wrong: {w0.grad.item():.4f}'
assert abs(x0.grad.item() - 0.393) < 0.01, f'df/dx0 wrong: {x0.grad.item():.4f}'
assert abs(w1.grad.item() - (-0.393)) < 0.01, f'df/dw1 wrong: {w1.grad.item():.4f}'
assert abs(x1.grad.item() - (-0.590)) < 0.01, f'df/dx1 wrong: {x1.grad.item():.4f}'
assert abs(w2.grad.item() - 0.197) < 0.01, f'df/dw2 wrong: {w2.grad.item():.4f}'
print('Test 5.1 PASSED: Backpropagation values are correct.')

Test 5.1 PASSED: Backpropagation values are correct.


### Task 5.2: Manual backward pass through a two-layer network

Given `x → (W1, b1) → ReLU → (W2, b2) → scores`, implement the backward pass **manually** (without autograd) and verify your gradients match PyTorch's.

**Steps:**
1. Compute forward pass manually (store intermediates)
2. Compute gradients of `scores[0]` w.r.t. W1, b1, W2, b2
3. Verify against `.backward()`

In [ ]:

torch.manual_seed(0)
x_bp = torch.randn(4)
W1_bp = torch.randn(3, 4, requires_grad=True)
b1_bp = torch.randn(3, requires_grad=True)
W2_bp = torch.randn(2, 3, requires_grad=True)
b2_bp = torch.randn(2, requires_grad=True)

# Step 1: Manual forward pass
z1 = W1_bp @ x_bp + b1_bp
a1 = torch.clamp(z1, min=0)
scores = W2_bp @ a1 + b2_bp

# Step 2: Manual backward pass (gradient of scores[0])
dscores = torch.zeros(2)
dscores[0] = 1.0

dL_dW2_manual = dscores.unsqueeze(1) * a1.unsqueeze(0)
dL_db2_manual = dscores

da1 = W2_bp.t() @ dscores
dz1 = da1 * (z1 > 0).float()

dL_dW1_manual = dz1.unsqueeze(1) * x_bp.unsqueeze(0)
dL_db1_manual = dz1

# Step 3: Autograd verification
scores_auto = W2_bp @ torch.clamp(W1_bp @ x_bp + b1_bp, min=0) + b2_bp
scores_auto[0].backward()

print("W2 grad match:", torch.allclose(dL_dW2_manual, W2_bp.grad))
print("b2 grad match:", torch.allclose(dL_db2_manual, b2_bp.grad))
print("W1 grad match:", torch.allclose(dL_dW1_manual, W1_bp.grad))
print("b1 grad match:", torch.allclose(dL_db1_manual, b1_bp.grad))




W2 grad match: True
b2 grad match: True
W1 grad match: True
b1 grad match: True


---
## Part 6: Activation Functions and Dead Neurons

Activation functions introduce non-linearity. The choice of activation affects gradient flow, training speed, and whether neurons "die" (become permanently inactive).

### Task 6.1: The dead ReLU problem

The network below has large negative biases that push many pre-activations below zero.

Pass 1000 test samples through each model (no training!) and count the fraction of neurons that output exactly zero for *all* samples. Compare ReLU vs Leaky ReLU.

In [ ]:
class MLPWithActivationTracking(nn.Module):
    def __init__(self, activation='relu'):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(3072, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 10)
        if activation == 'relu':
            self.act1 = nn.ReLU(); self.act2 = nn.ReLU()
        elif activation == 'leakyrelu':
            self.act1 = nn.LeakyReLU(0.01); self.act2 = nn.LeakyReLU(0.01)
        nn.init.normal_(self.fc1.weight, std=0.1)
        nn.init.constant_(self.fc1.bias, -2.0)
        nn.init.normal_(self.fc2.weight, std=0.1)
        nn.init.constant_(self.fc2.bias, -2.0)
        self.activations = {}
    def forward(self, x):
        x = self.flatten(x)
        x = self.act1(self.fc1(x))
        self.activations['layer1'] = x.detach()
        x = self.act2(self.fc2(x))
        self.activations['layer2'] = x.detach()
        return self.fc3(x)

In [ ]:
# 1. Create MLPWithActivationTracking('relu') and ('leakyrelu') — do NOT train
# 2. Pass 1000 test samples through each (DataLoader with batch_size=1000)
# 3. A neuron is "dead" if output < 1e-7 for ALL 1000 samples
# 4. Store fraction in dead_frac_relu and dead_frac_leaky

model_relu = MLPWithActivationTracking('relu').to(device)
model_relu.eval()
loader = DataLoader(testset_raw, batch_size=1000, shuffle=False)
inputs, _ = next(iter(loader))
with torch.no_grad():
    model_relu(inputs.to(device))
acts_relu = torch.cat([model_relu.activations['layer1'], model_relu.activations['layer2']], dim=1)
dead_frac_relu = (acts_relu.abs() < 1e-7).all(dim=0).float().mean().item()

model_leaky = MLPWithActivationTracking('leakyrelu').to(device)
model_leaky.eval()
with torch.no_grad():
    model_leaky(inputs.to(device))
acts_leaky = torch.cat([model_leaky.activations['layer1'], model_leaky.activations['layer2']], dim=1)
dead_frac_leaky = (acts_leaky.abs() < 1e-7).all(dim=0).float().mean().item()


print(f'Dead neurons ReLU: {dead_frac_relu:.1%}')
print(f'Dead neurons LeakyReLU: {dead_frac_leaky:.1%}')

Dead neurons ReLU: 21.0%
Dead neurons LeakyReLU: 0.0%


In [ ]:
assert dead_frac_relu is not None and dead_frac_leaky is not None
assert dead_frac_relu > dead_frac_leaky, \
    f'Expected ReLU ({dead_frac_relu:.1%}) to have more dead neurons than LeakyReLU ({dead_frac_leaky:.1%})'
print(f'Test 6.1 PASSED: ReLU dead={dead_frac_relu:.1%}, LeakyReLU dead={dead_frac_leaky:.1%}.')

Test 6.1 PASSED: ReLU dead=21.0%, LeakyReLU dead=0.0%.


### Task 6.2: Activation function comparison: Sigmoid vs. ReLU vs. Tanh

Train three models (20 epochs each, Adam lr=0.001, default init) with:
1. **Sigmoid** — 2. **ReLU** — 3. **Tanh**

Define `FlexibleMLP` with configurable activation. Use `copy.deepcopy(activation_fn)` for separate instances per layer.

In [ ]:

class FlexibleMLP(nn.Module):
    def __init__(self, activation_fn=nn.ReLU()):
        super().__init__()
        # Flatten → Linear(3072,512) → act → Linear(512,256) → act → Linear(256,10)
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(3072, 512),
            copy.deepcopy(activation_fn),
            nn.Linear(512, 256),
            copy.deepcopy(activation_fn),
            nn.Linear(256, 10),
        )

    def forward(self, x):
        return self.net(x)

criterion = nn.CrossEntropyLoss()

model_sig = FlexibleMLP(nn.Sigmoid()).to(device)
history_sigmoid_fair = train_and_log(model_sig, trainloader, testloader, criterion,
                                     optim.Adam(model_sig.parameters(), lr=0.001), 20, device)

model_relu = FlexibleMLP(nn.ReLU()).to(device)
history_relu_fair = train_and_log(model_relu, trainloader, testloader, criterion,
                                  optim.Adam(model_relu.parameters(), lr=0.001), 20, device)

model_tanh = FlexibleMLP(nn.Tanh()).to(device)
history_tanh_fair = train_and_log(model_tanh, trainloader, testloader, criterion,
                                  optim.Adam(model_tanh.parameters(), lr=0.001), 20, device)

# plot_multiple_histories([history_sigmoid_fair, history_relu_fair, history_tanh_fair],
                        # ['Sigmoid', 'ReLU', 'Tanh'], title='Activation Function Comparison')




Epoch   1/20 | Train Loss 1.8879 Acc 31.7% | Val Loss 1.8684 Acc 34.4% | LR 0.001000
Epoch   3/20 | Train Loss 1.6748 Acc 39.3% | Val Loss 1.8459 Acc 35.7% | LR 0.001000
Epoch   5/20 | Train Loss 1.6101 Acc 42.1% | Val Loss 1.7859 Acc 36.5% | LR 0.001000
Epoch   7/20 | Train Loss 1.5805 Acc 43.0% | Val Loss 1.7718 Acc 38.1% | LR 0.001000
Epoch   9/20 | Train Loss 1.5532 Acc 43.7% | Val Loss 1.7140 Acc 39.0% | LR 0.001000
Epoch  11/20 | Train Loss 1.5270 Acc 45.0% | Val Loss 1.7013 Acc 40.0% | LR 0.001000
Epoch  13/20 | Train Loss 1.5100 Acc 45.6% | Val Loss 1.6967 Acc 40.3% | LR 0.001000
Epoch  15/20 | Train Loss 1.4887 Acc 46.3% | Val Loss 1.6480 Acc 41.9% | LR 0.001000
Epoch  17/20 | Train Loss 1.4732 Acc 46.9% | Val Loss 1.6456 Acc 41.6% | LR 0.001000
Epoch  19/20 | Train Loss 1.4641 Acc 47.3% | Val Loss 1.6343 Acc 41.6% | LR 0.001000
Epoch  20/20 | Train Loss 1.4584 Acc 47.5% | Val Loss 1.6426 Acc 42.3% | LR 0.001000
Epoch   1/20 | Train Loss 1.7979 Acc 34.9% | Val Loss 1.6962 Acc 

In [ ]:
assert all(h is not None for h in [history_sigmoid_fair, history_relu_fair, history_tanh_fair])
plot_multiple_histories(
    [history_sigmoid_fair, history_relu_fair, history_tanh_fair],
    ['Sigmoid', 'ReLU', 'Tanh'],
    'Activation Function Comparison'
)
print('Test 6.2 PASSED.')

<Figure size 1400x500 with 2 Axes>

Test 6.2 PASSED.


### Task 6.3: Visualize activation distributions

For each of the three trained models, pass a test batch through and record the output of the **first hidden layer** (after activation). Plot three histograms side by side.

**Hint:** Use `register_forward_hook` on the first activation layer.

In [ ]:

def get_first_hidden_activations(model, inputs):
    activations = {}
    # net[2] is the first activation layer (after Flatten, Linear, *Activation*)
    hook = model.net[2].register_forward_hook(
        lambda module, inp, out: activations.update({'act': out.detach().cpu()})
    )
    model.eval()
    with torch.no_grad():
        model(inputs.to(device))
    hook.remove()
    return activations['act']

# Grab one test batch
test_imgs, _ = next(iter(testloader))

act_sigmoid = get_first_hidden_activations(model_sig, test_imgs)
act_relu = get_first_hidden_activations(model_relu, test_imgs)
act_tanh = get_first_hidden_activations(model_tanh, test_imgs)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, act, name in zip(axes, [act_sigmoid, act_relu, act_tanh], ['Sigmoid', 'ReLU', 'Tanh']):
    ax.hist(act.flatten().numpy(), bins=100, alpha=0.7)
    ax.set_title(f'{name} — Layer 1 Activations')
    ax.set_xlabel('Activation value')
    ax.set_ylabel('Count')
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()



<Figure size 1500x400 with 3 Axes>

---
## Part 7: Weight Initialization: Why It Matters

Bad initialization can cause activations to vanish or explode, making training impossible.

In [ ]:
class DeepMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(3072, 1024), nn.ReLU(),
            nn.Linear(1024, 512), nn.ReLU(),
            nn.Linear(512, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 10)
        )
    def forward(self, x): return self.net(x)

def apply_init(model, init_name):
    for m in model.modules():
        if isinstance(m, nn.Linear):
            if init_name == 'zeros': nn.init.zeros_(m.weight)
            elif init_name == 'large_random': nn.init.normal_(m.weight, std=1.0)
            elif init_name == 'small_random': nn.init.normal_(m.weight, std=0.001)
            elif init_name == 'kaiming': nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
            elif init_name == 'xavier': nn.init.xavier_normal_(m.weight)
            if m.bias is not None: nn.init.zeros_(m.bias)
    return model

### Task 7.1: Visualize forward pass statistics before training

**Without training**, pass one batch through `DeepMLP` and record the **standard deviation of activations** at each ReLU layer for four initializations: `zeros`, `large_random`, `small_random`, `kaiming`.

Plot activation std vs. layer index (use log scale for y-axis).

**Steps:**
1. Implement `get_activation_stats(model, dataloader, device)` — register hooks on ReLU layers, run one batch, return per-layer `(mean, std)`
2. For each init strategy, create a fresh `DeepMLP`, apply init, get stats
3. Plot std per layer

In [ ]:

init_strategies = ['zeros', 'large_random', 'small_random', 'kaiming']

def get_activation_stats(model, dataloader, device):
    """Run one batch, return per-ReLU-layer (mean, std) as a list."""
    model.to(device)
    model.eval()
    stats = []
    hooks = []

    for module in model.net:
        if isinstance(module, nn.ReLU):
            def make_hook(layer_stats):
                def hook_fn(mod, inp, out):
                    layer_stats.append((out.detach().mean().item(), out.detach().std().item()))
                return hook_fn
            layer_stat = []
            stats.append(layer_stat)
            hooks.append(module.register_forward_hook(make_hook(layer_stat)))

    inputs, _ = next(iter(dataloader))
    with torch.no_grad():
        model(inputs.to(device))

    for h in hooks:
        h.remove()

    return [(s[0][0], s[0][1]) for s in stats]


# Plot std per layer for each init
fig, ax = plt.subplots(figsize=(8, 5))
for init_name in init_strategies:
    model = apply_init(DeepMLP(), init_name)
    layer_stats = get_activation_stats(model, testloader, device)
    stds = [s[1] for s in layer_stats]
    ax.plot(range(len(stds)), stds, marker='o', label=init_name)

ax.set_yscale('log')
ax.set_xlabel('ReLU Layer Index')
ax.set_ylabel('Activation Std (log scale)')
ax.set_title('Activation Std vs. Layer for Different Initializations')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()



<Figure size 800x500 with 1 Axes>

In [ ]:
model_k = apply_init(DeepMLP(), 'kaiming').to(device)
stats_k = get_activation_stats(model_k, testloader, device)
assert stats_k is not None and len(stats_k) >= 3
assert stats_k[0][1] > 0.01 and stats_k[-1][1] > 0.01, 'Kaiming should keep stds reasonable.'
model_z = apply_init(DeepMLP(), 'zeros').to(device)
stats_z = get_activation_stats(model_z, testloader, device)
assert stats_z[-1][1] < 0.01, 'Zero init should produce near-zero activations.'
print('Test 7.1 PASSED.')

Test 7.1 PASSED.


### Task 7.2: Xavier vs Kaiming with different activations

Train four models (15 epochs, Adam lr=0.001):

| | Tanh | ReLU |
|---|---|---|
| Xavier | ✓ | ✓ |
| Kaiming | ✓ | ✓ |

Create `DeepMLPCustom(activation)` that accepts `'relu'` or `'tanh'`. Store histories in `init_act_histories` dict with keys like `'Xavier+tanh'`.

In [ ]:

class DeepMLPCustom(nn.Module):
    def __init__(self, activation='relu'):
        super().__init__()
        act = nn.ReLU if activation == 'relu' else nn.Tanh
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(3072, 1024), act(),
            nn.Linear(1024, 512), act(),
            nn.Linear(512, 256), act(),
            nn.Linear(256, 128), act(),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.net(x)


init_act_histories = {}
criterion = nn.CrossEntropyLoss()

configs = [
    ('Xavier', 'tanh', 'xavier'),
    ('Xavier', 'relu', 'xavier'),
    ('Kaiming', 'tanh', 'kaiming'),
    ('Kaiming', 'relu', 'kaiming'),
]

for init_label, activation, init_name in configs:
    key = f'{init_label}+{activation}'
    model = apply_init(DeepMLPCustom(activation), init_name).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    init_act_histories[key] = train_and_log(
        model, trainloader, testloader, criterion, optimizer, 15, device
    )

plot_multiple_histories(
    list(init_act_histories.values()),
    list(init_act_histories.keys()),
    title='Xavier vs Kaiming with Different Activations'
)



Epoch   1/15 | Train Loss 1.9247 Acc 30.1% | Val Loss 1.9832 Acc 30.0% | LR 0.001000
Epoch   2/15 | Train Loss 1.8018 Acc 34.8% | Val Loss 1.9505 Acc 32.9% | LR 0.001000
Epoch   3/15 | Train Loss 1.7518 Acc 36.9% | Val Loss 1.9773 Acc 31.0% | LR 0.001000
Epoch   4/15 | Train Loss 1.7334 Acc 37.6% | Val Loss 1.9133 Acc 34.2% | LR 0.001000
Epoch   5/15 | Train Loss 1.7079 Acc 38.3% | Val Loss 1.8988 Acc 34.0% | LR 0.001000
Epoch   6/15 | Train Loss 1.6932 Acc 38.9% | Val Loss 1.8736 Acc 35.9% | LR 0.001000
Epoch   7/15 | Train Loss 1.6795 Acc 39.6% | Val Loss 1.9205 Acc 33.8% | LR 0.001000
Epoch   8/15 | Train Loss 1.6676 Acc 39.9% | Val Loss 1.8588 Acc 35.1% | LR 0.001000
Epoch   9/15 | Train Loss 1.6512 Acc 40.8% | Val Loss 1.8383 Acc 36.7% | LR 0.001000
Epoch  10/15 | Train Loss 1.6485 Acc 40.7% | Val Loss 1.8378 Acc 35.9% | LR 0.001000
Epoch  11/15 | Train Loss 1.6396 Acc 41.1% | Val Loss 1.7858 Acc 37.5% | LR 0.001000
Epoch  12/15 | Train Loss 1.6322 Acc 41.4% | Val Loss 1.8110 Acc 

<Figure size 1400x500 with 2 Axes>

In [ ]:
assert len(init_act_histories) == 4, f'Expected 4 configs, got {len(init_act_histories)}'
plot_multiple_histories(list(init_act_histories.values()), list(init_act_histories.keys()),
                       'Xavier vs Kaiming × Tanh vs ReLU')
for name, h in init_act_histories.items():
    print(f'{name:20s} — final val acc: {h["val_acc"][-1]:.1f}%')
print('Test 7.2 PASSED.')

<Figure size 1400x500 with 2 Axes>

Xavier+tanh          — final val acc: 36.3%
Xavier+relu          — final val acc: 47.8%
Kaiming+tanh         — final val acc: 36.5%
Kaiming+relu         — final val acc: 48.1%
Test 7.2 PASSED.


---
## Part 8: Batch Normalization

Batch Normalization normalizes activations layer-by-layer, smoothing the loss landscape and enabling higher learning rates.

### Task 8.1: Define DeepMLPWithBN and test batch size sensitivity

Define `DeepMLPWithBN` — same architecture as `DeepMLP` but with `BatchNorm1d` after each Linear (before ReLU).

Train with three batch sizes: **4, 32, 128** (15 epochs each, Adam lr=0.001). Store in `bn_batch_histories` dict keyed by batch size.

**Why this matters:** BN computes mean/variance over the batch — with very small batches, these statistics become noisy.

In [ ]:

class DeepMLPWithBN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(3072, 1024), nn.BatchNorm1d(1024), nn.ReLU(),
            nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.ReLU(),
            nn.Linear(512, 256), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.net(x)


batch_sizes_to_test = [4, 32, 128]
bn_batch_histories = {}
criterion = nn.CrossEntropyLoss()

for bs in batch_sizes_to_test:
    loader_train = DataLoader(trainset, batch_size=bs, shuffle=True)
    model = DeepMLPWithBN().to(device)
    apply_init(model, 'kaiming')
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    bn_batch_histories[bs] = train_and_log(
        model, loader_train, testloader, criterion, optimizer, 15, device
    )

plot_multiple_histories(
    list(bn_batch_histories.values()),
    [f'BS={bs}' for bs in batch_sizes_to_test],
    title='Batch Normalization — Batch Size Sensitivity'
)



Epoch   1/15 | Train Loss 2.1489 Acc 21.0% | Val Loss 1.9150 Acc 30.3% | LR 0.001000
Epoch   2/15 | Train Loss 2.0513 Acc 25.2% | Val Loss 1.8815 Acc 35.7% | LR 0.001000
Epoch   3/15 | Train Loss 2.0097 Acc 26.9% | Val Loss 1.8436 Acc 36.1% | LR 0.001000
Epoch   4/15 | Train Loss 1.9793 Acc 28.6% | Val Loss 1.7960 Acc 37.8% | LR 0.001000
Epoch   5/15 | Train Loss 1.9636 Acc 28.9% | Val Loss 1.8028 Acc 39.2% | LR 0.001000
Epoch   6/15 | Train Loss 1.9493 Acc 29.9% | Val Loss 1.7904 Acc 38.2% | LR 0.001000
Epoch   7/15 | Train Loss 1.9329 Acc 30.4% | Val Loss 1.7431 Acc 41.0% | LR 0.001000
Epoch   8/15 | Train Loss 1.9209 Acc 30.9% | Val Loss 1.7308 Acc 41.5% | LR 0.001000
Epoch   9/15 | Train Loss 1.9111 Acc 31.3% | Val Loss 1.7087 Acc 42.1% | LR 0.001000
Epoch  10/15 | Train Loss 1.9002 Acc 32.0% | Val Loss 1.7104 Acc 42.5% | LR 0.001000
Epoch  11/15 | Train Loss 1.8917 Acc 32.5% | Val Loss 1.6631 Acc 42.7% | LR 0.001000
Epoch  12/15 | Train Loss 1.8795 Acc 32.6% | Val Loss 1.7131 Acc 

<Figure size 1400x500 with 2 Axes>

In [ ]:
assert len(bn_batch_histories) == 3
for bs in batch_sizes_to_test:
    print(f'Batch size {bs:3d} — final val acc: {bn_batch_histories[bs]["val_acc"][-1]:.1f}%')
plot_multiple_histories(
    [bn_batch_histories[bs] for bs in batch_sizes_to_test],
    [f'BS={bs}' for bs in batch_sizes_to_test],
    'BN Batch Size Sensitivity'
)
print('Test 8.1 PASSED.')

Batch size   4 — final val acc: 43.2%
Batch size  32 — final val acc: 55.8%
Batch size 128 — final val acc: 57.4%


<Figure size 1400x500 with 2 Axes>

Test 8.1 PASSED.


### Task 8.2: Train vs. eval mode — a common bug

BN behaves differently in `.train()` vs `.eval()` mode:
- **Train mode:** Uses batch statistics (mean/var of current batch)
- **Eval mode:** Uses running statistics (accumulated during training)

Train a `DeepMLPWithBN` for 15 epochs, then evaluate accuracy in both modes. Store in `acc_eval_mode` and `acc_train_mode`.

**Think about:** Why might accuracy differ? Which is correct for deployment?

In [ ]:

model_bn = DeepMLPWithBN().to(device)
apply_init(model_bn, 'kaiming')
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_bn.parameters(), lr=0.001)
_ = train_and_log(model_bn, trainloader, testloader, criterion, optimizer, 15, device)

# Evaluate in eval mode (running statistics)
model_bn.eval()
_, acc_eval_mode = evaluate(model_bn, testloader, criterion, device)

# Evaluate in train mode (batch statistics)
model_bn.train()
_, acc_train_mode = evaluate(model_bn, testloader, criterion, device)


print(f'Accuracy with model.eval(): {acc_eval_mode:.1f}%')
print(f'Accuracy with model.train(): {acc_train_mode:.1f}%')

Epoch   1/15 | Train Loss 1.7894 Acc 35.3% | Val Loss 1.5965 Acc 42.5% | LR 0.001000
Epoch   2/15 | Train Loss 1.6043 Acc 42.1% | Val Loss 1.5103 Acc 45.3% | LR 0.001000
Epoch   3/15 | Train Loss 1.5276 Acc 44.5% | Val Loss 1.4212 Acc 48.3% | LR 0.001000
Epoch   4/15 | Train Loss 1.4805 Acc 46.6% | Val Loss 1.3771 Acc 49.9% | LR 0.001000
Epoch   5/15 | Train Loss 1.4394 Acc 48.1% | Val Loss 1.3412 Acc 51.3% | LR 0.001000
Epoch   6/15 | Train Loss 1.4044 Acc 49.4% | Val Loss 1.3354 Acc 52.2% | LR 0.001000
Epoch   7/15 | Train Loss 1.3820 Acc 50.3% | Val Loss 1.3134 Acc 53.5% | LR 0.001000
Epoch   8/15 | Train Loss 1.3530 Acc 51.1% | Val Loss 1.2893 Acc 54.1% | LR 0.001000
Epoch   9/15 | Train Loss 1.3341 Acc 51.9% | Val Loss 1.2750 Acc 54.3% | LR 0.001000
Epoch  10/15 | Train Loss 1.3163 Acc 52.6% | Val Loss 1.2727 Acc 54.9% | LR 0.001000
Epoch  11/15 | Train Loss 1.2995 Acc 53.4% | Val Loss 1.2447 Acc 55.3% | LR 0.001000
Epoch  12/15 | Train Loss 1.2789 Acc 53.8% | Val Loss 1.2385 Acc 

---
## Part 9: Dropout and Regularization

Dropout randomly zeroes neurons during training, forcing the network to learn redundant representations.

### Task 9.1: Dropout experiment: finding the sweet spot

Define `DeepMLPWithDropout`: same as `DeepMLPWithBN` but with `Dropout(rate)` after each ReLU.

Train four models (30 epochs, Adam lr=0.001) with dropout rates: **0.0, 0.2, 0.5, 0.8**. Store in `dropout_histories` dict.

In [ ]:

class DeepMLPWithDropout(nn.Module):
    def __init__(self, dropout_rate=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(3072, 1024), nn.BatchNorm1d(1024), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(512, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.net(x)


dropout_rates = [0.0, 0.2, 0.5, 0.8]
dropout_histories = {}
criterion = nn.CrossEntropyLoss()

for dr in dropout_rates:
    model = DeepMLPWithDropout(dr).to(device)
    apply_init(model, 'kaiming')
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    dropout_histories[dr] = train_and_log(
        model, trainloader, testloader, criterion, optimizer, 30, device
    )

plot_multiple_histories(
    list(dropout_histories.values()),
    [f'Dropout={dr}' for dr in dropout_rates],
    title='Dropout Rate Comparison'
)



Epoch   1/30 | Train Loss 1.7992 Acc 34.8% | Val Loss 1.6093 Acc 42.3% | LR 0.001000
Epoch   4/30 | Train Loss 1.4805 Acc 46.6% | Val Loss 1.3966 Acc 49.7% | LR 0.001000
Epoch   7/30 | Train Loss 1.3829 Acc 50.1% | Val Loss 1.3110 Acc 53.1% | LR 0.001000
Epoch  10/30 | Train Loss 1.3125 Acc 52.7% | Val Loss 1.2758 Acc 54.2% | LR 0.001000
Epoch  13/30 | Train Loss 1.2607 Acc 54.5% | Val Loss 1.2186 Acc 56.9% | LR 0.001000
Epoch  16/30 | Train Loss 1.2113 Acc 56.5% | Val Loss 1.2114 Acc 57.2% | LR 0.001000
Epoch  19/30 | Train Loss 1.1744 Acc 57.9% | Val Loss 1.1801 Acc 58.2% | LR 0.001000
Epoch  22/30 | Train Loss 1.1378 Acc 59.3% | Val Loss 1.1430 Acc 59.2% | LR 0.001000
Epoch  25/30 | Train Loss 1.1035 Acc 60.2% | Val Loss 1.1356 Acc 59.9% | LR 0.001000
Epoch  28/30 | Train Loss 1.0715 Acc 61.6% | Val Loss 1.1189 Acc 60.4% | LR 0.001000
Epoch  30/30 | Train Loss 1.0507 Acc 62.3% | Val Loss 1.1213 Acc 60.2% | LR 0.001000
Epoch   1/30 | Train Loss 1.9517 Acc 29.2% | Val Loss 1.6404 Acc 

In [ ]:
assert len(dropout_histories) == 4
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for dr in dropout_rates:
    h = dropout_histories[dr]
    ax1.plot(h['val_acc'], label=f'Dropout={dr}')
    gap = [t - v for t, v in zip(h['train_acc'], h['val_acc'])]
    ax2.plot(gap, label=f'Dropout={dr}')
ax1.set_title('Validation Accuracy'); ax1.legend(); ax1.grid(True, alpha=0.3)
ax2.set_title('Generalization Gap (Train - Val)'); ax2.legend(); ax2.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
for dr in dropout_rates:
    h = dropout_histories[dr]
    print(f'Dropout={dr:.1f} — Train: {h["train_acc"][-1]:.1f}%, Val: {h["val_acc"][-1]:.1f}%')
print('Test 9.1 PASSED.')

<Figure size 1400x500 with 2 Axes>

Dropout=0.0 — Train: 62.3%, Val: 60.2%
Dropout=0.2 — Train: 54.5%, Val: 57.9%
Dropout=0.5 — Train: 46.2%, Val: 53.4%
Dropout=0.8 — Train: 27.1%, Val: 32.9%
Test 9.1 PASSED.


---
## Part 10: Best MLP Challenge

You now have all the building blocks: architecture design, loss functions, optimizers, activation functions, initialization, batch normalization, dropout, and data augmentation. This final part combines everything to build the best MLP possible on CIFAR-10.

### The Scientific Approach

Rather than guessing randomly, treat this as a systematic engineering exercise:

1. **Start with a strong baseline** — use what was learned in Parts 6-9 (BN + Dropout + Kaiming init + a good activation).
2. **Change one thing at a time** — this is called an *ablation study*. It's how real ML research isolates what works.
3. **Log everything** — record what was tried, what worked, and what didn't.

### Techniques to consider

- **Architecture:** Wider layers, deeper networks, skip/residual connections
- **Activation:** ReLU, ELU, LeakyReLU
- **Optimizer:** Adam, SGD+Momentum, AdamW (with weight decay)
- **Learning rate scheduling:** `CosineAnnealingLR`, `ReduceLROnPlateau`, warmup
  ```python
  scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
  history = train_and_log(model, trainloader, testloader, criterion, optimizer, epochs, device, scheduler=scheduler)
  ```
- **Regularization:** Dropout rate, weight decay, data augmentation strength
- **Training duration:** More epochs with LR decay often helps

### Why this approach scales

This systematic methodology — baseline, controlled experiments, documentation — is exactly how ML teams develop models in practice. A well-documented negative result ("tried X, it didn't help because Y") is often as valuable as a raw accuracy number.

No convolutional layers allowed for this part; the goal is to see how far a pure MLP can go before hitting its architectural ceiling.


### Designing, training, and documenting the best MLP

**Deliverables:**
1. A `BestMLPModel` class (no Conv layers)
2. Training code that produces `history_best`
3. A detailed **writeup** (in the markdown cell below) documenting:
   - Baseline and final architecture
   - Each experiment tried (what, why, result)
   - Training recipe (optimizer, LR schedule, epochs, augmentation)
   - What worked, what didn't, and hypotheses for why
   - Why MLPs have a fundamental accuracy ceiling on CIFAR-10


In [ ]:
class BestMLPModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(3072, 1024), nn.BatchNorm1d(1024), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(512, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.net(x)

best_model = BestMLPModel().to(device)
apply_init(best_model, 'kaiming')
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(best_model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=80)

history_best = train_and_log(best_model, trainloader, testloader, criterion, optimizer, 80, device, scheduler=scheduler)




Epoch   1/80 | Train Loss 1.9451 Acc 29.7% | Val Loss 1.6607 Acc 40.0% | LR 0.001000
Epoch   9/80 | Train Loss 1.4750 Acc 47.1% | Val Loss 1.3338 Acc 52.2% | LR 0.000976
Epoch  17/80 | Train Loss 1.3703 Acc 50.8% | Val Loss 1.2318 Acc 56.0% | LR 0.000905
Epoch  25/80 | Train Loss 1.2944 Acc 53.7% | Val Loss 1.1888 Acc 57.2% | LR 0.000794
Epoch  33/80 | Train Loss 1.2397 Acc 55.7% | Val Loss 1.1247 Acc 59.9% | LR 0.000655
Epoch  41/80 | Train Loss 1.1874 Acc 57.4% | Val Loss 1.0908 Acc 61.0% | LR 0.000500
Epoch  49/80 | Train Loss 1.1449 Acc 59.4% | Val Loss 1.0678 Acc 62.3% | LR 0.000345
Epoch  57/80 | Train Loss 1.1117 Acc 60.3% | Val Loss 1.0379 Acc 63.1% | LR 0.000206
Epoch  65/80 | Train Loss 1.0887 Acc 61.2% | Val Loss 1.0285 Acc 63.1% | LR 0.000095
Epoch  73/80 | Train Loss 1.0725 Acc 61.6% | Val Loss 1.0203 Acc 63.5% | LR 0.000024
Epoch  80/80 | Train Loss 1.0704 Acc 61.6% | Val Loss 1.0210 Acc 63.5% | LR 0.000000


In [ ]:
assert history_best is not None, 'Train your best model.'
for m in best_model.modules():
    assert not isinstance(m, (nn.Conv2d, nn.Conv1d)), 'No convolutional layers allowed!'
best_acc = max(history_best['val_acc']); final_acc = history_best['val_acc'][-1]
print(f'Best validation accuracy: {best_acc:.1f}%')
print(f'Final validation accuracy: {final_acc:.1f}%')
if best_acc >= 62: print('Accuracy grade: 5/5 — Excellent!')
elif best_acc >= 58: print('Accuracy grade: 3/5 — Great job.')
elif best_acc >= 54: print('Accuracy grade: 2/5 — Good effort.')
else: print(f'Accuracy grade: 1/5 — Keep tuning.')
plot_history(history_best, f'Best MLP ({best_acc:.1f}%)')
print('Test 10.1 PASSED.')

Best validation accuracy: 63.7%
Final validation accuracy: 63.5%
Accuracy grade: 5/5 — Excellent!


<Figure size 1400x500 with 2 Axes>

Test 10.1 PASSED.


### Writeup

*Documenting the approach below.*

**Baseline:**
- Architecture: 3072→1024→512→256→128→10 with BatchNorm + ReLU + Dropout(0.2) after each hidden layer
- Optimizer: Adam (lr=0.001)
- Initialization: Kaiming normal
- Loss: CrossEntropyLoss
- Epochs: 30
- Data augmentation: RandomCrop(32, padding=4) + RandomHorizontalFlip (from Task 1.3)
- Result: 58.8% val accuracy, still improving at epoch 30

**Experiments tried:**

| # | What I changed | Why | Result (val acc) | Keep? |
|---|---------------|-----|-----------------|-------|
| 1 | Wider first layer 1024 → 2048 |Include more neurons for learning in the beginning | 58.8% → 58.6% | No |
| 2 | Increase to 50 epochs | Baseline was still improving at epoch 30, hadn't plateaued | 58.8% → 61.0% | Yes |
| 3 | CosineAnnealingLR | Smooth LR decay prevents overfitting over longer training | 61.0% → 61.4% | Yes |
| 4 | Increase to 80 epochs (T_max=80) | Model still improving; more epochs with proper cosine decay lets it converge further | 61.4% → 63.3% | Yes |
| 5 | AdamW (weight_decay=1e-4) | Decoupled weight decay regularizes better than standard Adam | 63.3% → 63.7% | Yes |


**Final recipe:**
- Architecture: 3072→1024→512→256→128→10 with BatchNorm + ReLU + Dropout(0.2) after each hidden layer
- Optimizer: AdamW (lr=0.001, weight_decay=1e-4)
- LR schedule: CosineAnnealingLR (T_max=80)
- Epochs: 80
- Initialization: Kaiming normal
- Loss: CrossEntropyLoss
- Data augmentation: RandomCrop(32, padding=4) + RandomHorizontalFlip
- Best val accuracy: 63.7%

**What didn't work and why:**
- Wider first layer (1024→2048): No improvement, this suggested the bottleneck was not model capacity at the input stage but rather training duration and optimization. No benefit to the added costs.
- No LR schedule: Constant learning rate caused the model to plateau earlier. The cosine schedule allowed the model to make large updates early and fine-tune later as the rate decayed toward zero.


**Why MLPs have a ceiling on CIFAR-10:**
MLPs flatten the image into a 1D vector, destroying all spatial structure. As noted in "Understanding Machine Learning: From Theory to Algorithms" (2014, Section 25.3, Feature Engineering), "applying a linear predictor on the pixel-based representation of the image does not yield a good classifier" — the raw pixel representation lacks the meaningful features needed for classification. An MLP must learn to extract these "visual words" (edges, textures, shapes) entirely from scratch using fully-connected layers, where every pixel is connected to every neuron regardless of spatial proximity. This means a cat's ear in the top-left requires completely different weights than the same ear in the bottom-right. CNNs solve this by building in spatial inductive biases — local receptive fields that detect features in small neighborhoods, weight sharing that makes those detections translation-invariant, and hierarchical pooling that builds complex features from simple ones. An MLP has to discover all of this structure from data alone, which is why even a well-tuned MLP plateaus around 60-65% on CIFAR-10 while simple CNNs easily exceed 90%.


---
### Summary

In [ ]:
print('=' * 60)
print('PROJECT SUMMARY')
print('=' * 60)
results = []
if 'history_fixed_linear' in dir() and history_fixed_linear:
    results.append(('Linear Classifier', history_fixed_linear['val_acc'][-1]))
if 'history_mlp' in dir() and history_mlp:
    results.append(('MLP', history_mlp['val_acc'][-1]))
if 'history_best' in dir() and history_best:
    results.append(('Best MLP Model', max(history_best['val_acc'])))
for name, acc in results:
    print(f'{name:30} {acc:11.1f}%')
print('=' * 60)
print('END OF PROJECT')